In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.7 MB/s eta 0:00:00


In [5]:
import gc
import torch

# Удалить модель и освободить память
del llm  # Удалить объект модели vLLM
del tokenizer  # Удалить токенизатор

# Запустить сборщик мусора Python
gc.collect()

# Очистить кеш CUDA
torch.cuda.empty_cache()

# Проверить освобождённую память
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")


GPU memory allocated: 0.00 GB
GPU memory reserved: 0.00 GB


In [2]:
!pip install -q transformers accelerate torch vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.2/438.2 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.0/180.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.4

In [3]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

MODEL_NAME = "IlyaGusev/saiga_llama3_8b"
MODEL_NAME = "Vikhrmodels/QVikhr-3-4B-Instruction"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

llm = LLM(
    model=MODEL_NAME,
    max_model_len=8192,
    gpu_memory_utilization=0.9,
    disable_custom_all_reduce=True,
    trust_remote_code=True,
    tensor_parallel_size=1,
    enable_chunked_prefill=True,
    max_num_seqs=256,
)

INFO 11-13 19:54:07 [__init__.py:216] Automatically detected platform cuda.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

INFO 11-13 19:54:58 [model.py:547] Resolved architecture: Qwen3ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-13 19:54:58 [model.py:1510] Using max model len 8192
INFO 11-13 19:55:01 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.


generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

INFO 11-13 19:59:02 [llm.py:306] Supported_tasks: ['generate']


In [4]:
SYSTEM_PROMPT = """
/no_think

Ты — AI‑ассистент, эксперт по 2‑уровневой классификации обращений клиентов авиакомпании; цель — присвоить строго одну категорию Level 1 и одну или несколько подкатегорий Level 2 из утверждённого справочника ниже, без выхода за рамки словаря.


Правила вывода:
- Возвращай только валидный JSON, без кода, подсказок, комментариев, разметки, пояснений, преамбул и суффиксов.
- Схема: {"level_1": "<строка>", "level_2": ["<строка>", ...]}.
- Значения берутся только из словаря категорий; новые категории не придумывать.
- "level_2" обязателен и не пустой, без дублей, порядок — по убыванию релевантности.
- Язык: русские названия ровно как в словаре (регистр и орфография совпадают).
- Если есть приветствие/прощание и содержательная тема, выбирай содержательную тему.
- Приоритет Level 1: Обратная связь (жалоба/эскалация) > Вылет Прилет > До Вылета > На борту > После вылета > Технические вопросы > Лояльность и продажи > Сервисные > Другое.
- В неопределённых случаях используй {"level_1":"Другое","level_2":["Другое"]}.
- Игнорируй ссылки, смайлы, подписи; распознавай коды рейсов, даты, ключевые синонимы.


СПРАВОЧНИК КАТЕГОРИЙ:


УРОВЕНЬ 1: ДО ВЫЛЕТА
Область: поиск, бронь, оплата, регистрация, документы, багаж до сдачи (ДО совершения полёта).
Ключевые сигналы: купить, забронировать, поменять дату, оплатить, регистрация, посадочный талон, паспорт, виза, норма багажа, перевес, животные.

Level 2:
- Бронирование: покупка/изменение маршрута, выбор места/багажа при покупке, промокоды, тарифы, изменение даты/ФИО.
- Оплата: платёж, ошибки платежей, двойное списание, возвраты денег до вылета, чеки.
- Регистрация: онлайн/стойка, посадочные талоны, сроки регистрации.
- Багаж (до вылета): нормы, перевес, доплата, спортинвентарь, животные, коляска.
- Документы: паспорта, визы, вакцинация, согласие на выезд, требования.


УРОВЕНЬ 1: ВЫЛЕТ ПРИЛЕТ
Область: статус рейса, задержки, отмены, нештатные ситуации в аэропорту, потерянный багаж, трансфер/гостиница при ИВР.
Ключевые сигналы: статус, задержка, отмена, гейт, пересадка, утерян багаж, ваучер, отель, трансфер.

Level 2:
- Статус рейса: время вылета/прилёта, гейт, информация о пересадке.
- Задержка рейса: причины, ваучеры на питание.
- Отмена рейса: пересадка/возврат из‑за отмены.
- Гостиница трансфер: размещение/трансфер при ИВР.
- Поиск багажа: утерян/задержан/повреждён багаж, PIR.


УРОВЕНЬ 1: НА БОРТУ
Область: обслуживание, чистота, комфорт, питание, развлечения, ручная кладь (ВО ВРЕМЯ полёта в салоне).
Ключевые сигналы: стюардесса, грязно, холодно/жарко, еда, Wi‑Fi, экран, ручная кладь, полка.

Level 2:
- Обслуживание борт: работа экипажа, внимание, помощь.
- Чистота борт: уборка, санитария.
- Комфорт борт: температура, шум, кресла, пледы, подушки.
- Питание борт: меню, качество, спецпитание (веган, детское).
- Развлечения борт: IFE, Wi‑Fi, экраны, контент.
- Багаж борт: ручная кладь в салоне, размещение на полках.


УРОВЕНЬ 1: ПОСЛЕ ВЫЛЕТА
Область: возвраты, обмены, компенсации после полёта, претензии, забытые вещи.
Ключевые сигналы: возврат денег, обмен, компенсация, EU261, расходы, гостиница, забыл вещи.

Level 2:
- Возврат билета: возврат средств, no‑show, условия тарифа.
- Обмен билета: изменение после полёта, доплаты.
- Компенсация: выплаты за ИВР, EU261, повреждения, расходы пассажира.
- Потерянная собственность: забытые вещи в салоне (не багажная лента).


УРОВЕНЬ 1: ЛОЯЛЬНОСТЬ И ПРОДАЖИ
Область: программа лояльности, маркетинг, акции, скидки.
Ключевые сигналы: мили, статус, программа, акция, скидка, промокод, тариф.

Level 2:
- Программа лояльности: мили, статусы, регистрация, начисление/списание, трансфер.
- Акции тарифы: скидки, промо, спецпредложения, тарифы.


УРОВЕНЬ 1: ТЕХНИЧЕСКИЕ ВОПРОСЫ
Область: цифровые каналы (сайт, приложение, платежи, чат‑бот).
Ключевые сигналы: ошибка, не работает, не загружается, crash, платёж, бот, приложение.

Level 2:
- Работа сайта приложения: ошибки, краши, логин, обновления, платежи.
- Работа бота: некорректные ответы, недоступность.


УРОВЕНЬ 1: ОБРАТНАЯ СВЯЗЬ
Область: метакоммуникация — жалобы (без денежных требований), благодарность, предложения, эскалация.
Ключевые сигналы: жалоба, претензия, спасибо, благодарность, предложение, руководитель, эскалация.

Level 2:
- Жалоба: негативная оценка, требование разбирательства.
- Благодарность: позитивная обратная связь, похвала.
- Предложение: идеи улучшений, пожелания.
- Эскалация: запрос связи с руководством, официальная претензия.


УРОВЕНЬ 1: СЕРВИСНЫЕ
Область: ритуальные фразы без предмета (приветствия, прощания).

Level 2:
- Приветствие: привет/добрый день без запроса.
- Прощание: благодарю/до свидания без запроса.


УРОВЕНЬ 1: ДРУГОЕ
Область: вне словаря, спам, реклама, неклассифицируемое.

Level 2:
- Другое: используй только если нет соответствия.


Правила дизамбигуации:
- Багаж ДО сдачи → «До Вылета/Багаж (до вылета)»; потеря/задержка → «Вылет Прилет/Поиск багажа»; ручная кладь → «На борту/Багаж борт».
- Обмен/возврат ДО полёта → «До Вылета»; ПОСЛЕ полёта → «После вылета».
- Жалоба + тема: если оценка/негатив → «Обратная связь/Жалоба»; если запрос действия → по теме.
- Приоритет: выбери доминирующую категорию по приоритету (см. правила вывода).


Примеры:
"Не могу оплатить билет, ошибка 403" → {"level_1":"До Вылета","level_2":["Оплата"]}
"Рейс SU123 задержан 3 часа, ваучер" → {"level_1":"Вылет Прилет","level_2":["Задержка рейса"]}
"На борту холодно, плед" → {"level_1":"На борту","level_2":["Комфорт борт"]}
"Чемодан потерялся" → {"level_1":"Вылет Прилет","level_2":["Поиск багажа"]}
"Спасибо бортпроводникам, как поменять билет?" → {"level_1":"После вылета","level_2":["Обмен билета"]}
"Добрый день!" → {"level_1":"Сервисные","level_2":["Приветствие"]}
"Паспорт истёк, лететь можно?" → {"level_1":"До Вылета","level_2":["Документы"]}
"Требую компенсацию за потерянный багаж" → {"level_1":"После вылета","level_2":["Компенсация"]}
"Wi‑Fi не работает на борту" → {"level_1":"На борту","level_2":["Развлечения борт"]}
"Приложение крашится" → {"level_1":"Технические вопросы","level_2":["Работа сайта приложения"]}
"Мили не начислились" → {"level_1":"Лояльность и продажи","level_2":["Программа лояльности"]}
"Ваш сервис ужасен, хочу с руководителем" → {"level_1":"Обратная связь","level_2":["Жалоба","Эскалация"]}


Формат ответа: Верни только JSON-объект по схеме выше в одной строке, без кода, маркдауна и пунктуации.
"""

In [5]:
sampling_params = SamplingParams(
    temperature=0.15,
    top_p=0.9,
    max_tokens=50,
    skip_special_tokens=True
)

In [6]:
def classify_text(user_texts: list[str]) -> dict:
    """
    Классифицирует входящий текст пользователя с помощью LLM.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True
    )[1:]
    text = tokenizer.decode(input_ids)

    outputs = llm.generate(text, use_tqdm=False, sampling_params=sampling_params)

    response_text = tokenizer.decode(outputs[0].outputs[0].token_ids, skip_special_tokens=True)


    try:
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        if json_start != -1 and json_end != -1:
            json_str = response_text[json_start:json_end]
            return json.loads(json_str)
        else:
            return {"error": "JSON object not found in the response", "raw_response": response_text}
    except json.JSONDecodeError:
        return {"error": "Failed to decode JSON", "raw_response": response_text}

In [7]:
def classify_texts_batch(user_texts: list[str]) -> list[dict]:
    """
    Классифицирует список текстов с использованием пакетной обработки vLLM.
    Намного быстрее, чем обработка по одному!
    """

    formatted_prompts = []

    for user_text in user_texts:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text}
        ]

        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True
        )[1:]

        prompt_text = tokenizer.decode(input_ids)
        formatted_prompts.append(prompt_text)

    outputs = llm.generate(formatted_prompts, sampling_params)

    results = []
    for output in outputs:
        response_text = tokenizer.decode(
            output.outputs[0].token_ids,
            skip_special_tokens=True
        )


        try:
            json_start = response_text.find('{')
            json_end = response_text.rfind('}') + 1

            if json_start != -1 and json_end > json_start:
                json_str = response_text[json_start:json_end]
                result = json.loads(json_str)
            else:
                result = {
                    "error": "JSON object not found in the response",
                    "raw_response": response_text
                }
        except json.JSONDecodeError:
            result = {
                "error": "Failed to decode JSON",
                "raw_response": response_text
            }

        results.append(result)

    return results

In [11]:
import json
test_data = [
    # До Вылета
    "Привет, я хочу купить билет на Москву в пятницу",
    "Как поменять дату моего билета? Летел на 15 января",
    "Нужен билет из SPB в MSK, когда у вас есть рейсы?",
    "Помогите выбрать место в самолёте, какой лучше?",
    "У вас есть скидки на билеты сейчас?",
    "Я хочу вернуть билет, деньги мне назад?",
    "Как применить промокод при покупке?",
    "Сколько багажа можно взять с собой?",
    "Паспорт истёк, я смогу летать?",
    "Нужна ли виза для Испании?",
    "Оплатить билет карточкой, это возможно?",
    "Деньги со счета списались дважды, помогите!",
    "Билет не пришел на почту после оплаты",
    "Как зарегистрироваться онлайн на рейс?",
    "Когда открывается регистрация?",
    "Я хочу выбрать место у окна",
    "Какую дополнительную услугу мне купить?",
    "Тариф эконом и комфорт - в чем разница?",
    "Нужно ли везти справку о вакцинации?",
    "Мне 2 года, нужен ли мне билет?",
    "Я хочу поменять ФИО в билете",
    "Можно ли платить в рассрочку?",
    "Вернул деньги в банк - правильно ли?",
    "Супруга не выезжает, вернёт ли билет?",
    "Рейс SU101 есть ли свободные места?",
    "В аэропорту какой можно припарковаться?",
    "Какой минимальный багаж включен?",
    "А если я везу спортивный инвентарь?",
    "Путешествую с кошкой, как её везти?",
    "Раскладная коляска можно взять?",
    "Требование по документам какое?",
    "Свидетельство рождения нужно?",
    "Согласие от второго родителя нужно ли?",
    "COVID справка нужна?",
    "Как забронировать через сайт?",
    "Почему билет стоит дороже чем вчера?",
    "А если я прошу место для инвалида?",
    "Срок действия паспорта когда заканчивается?",
    "Я беременна, проблем не будет?",
    "Какой размер ручной клади максимум?",
    "Летаю в первый раз, что нужно?",
    "Дешевле если брать туда-обратно?",
    "Изменить маршрут можно после покупки?",
    "Студент - есть скидки?",
    "Когда билеты дешевле всего?",
    "Система не принимает мою карту",
    "Два взрослых и ребёнок - цена какая?",
    "Можно ли купить билет за друга?",
    "Эконом плюс стоит ли брать?",
    "Компьютер не даёт мне забронировать",

    # Вылет Прилет
    "Статус рейса SU101, когда вилет?",
    "Мой рейс задержан, на сколько часов?",
    "Рейс отменили, что делать?",
    "Когда приступ в Питере?",
    "Какой gate на мой рейс?",
    "Я опоздал на пересадку, что дальше?",
    "Дайте мне ваучер на еду за задержку",
    "Рейс перенесли на какое время?",
    "Где стойка выдачи багажа?",
    "Чемодан не приехал, что делать?",
    "Мой багаж потеряется, помогите!",
    "Как открыть PIR на потерянный багаж?",
    "Нужен трансфер до отеля из-за задержки",
    "Где я могу остаться ночью?",
    "Рейс отменён, когда следующий?",
    "Гарантирована ли моя пересадка?",
    "Самолёт сломался, когда полетим?",
    "В какой отель вы меня разместите?",
    "Питание будет предоставлено при задержке?",
    "Когда можно забрать багаж?",
    "Чемодан повреждён при выдаче",
    "Я не в списке на рейс",
    "Информация на табло не обновляется",
    "Персонал не знает статус рейса",
    "Рейс из Москвы на Питер какой gate?",
    "Раньше ли вылет может быть?",
    "Как долго ждать на задержке?",
    "Я вот уже 5 часов жду",
    "Самолёт на земле уже час, почему?",
    "Это техническая проблема?",
    "Погода портит график?",
    "Сколько ещё придётся ждать?",
    "Это моя вина что опоздал?",
    "Вы ответственны за мои убытки?",
    "Чемодан прибыл со следующим рейсом",
    "Ехал ваш ваучер на гостиницу",
    "Официально я не зарегистрирован?",
    "Когда отправится альтернативный рейс?",
    "Я был на борту, самолёт развернулся",
    "Пассажир заболел, эвакуация?",
    "Воздушное сообщение закрыто?",
    "Забастовка персонала?",
    "Как часто это происходит?",
    "Мне хочется денег назад",
    "Я хочу на другой рейс",
    "Трансфер когда придёт?",
    "Багаж на каком рейсе прилетит?",
    "Это займёт очень долго?",
    "Почему никто не объясняет ситуацию?",
    "Я требую компенсацию!",

    # На борту
    "На борту очень холодно, дайте плед",
    "Стюардесса очень грубая была",
    "В салоне грязно, можно ли что-то сделать?",
    "Кресло сломано, не опускается спинка",
    "Еда невкусная и холодная",
    "Wi-Fi не работает совсем",
    "Экран в кресле не включается",
    "Наушники не подходят, есть другие?",
    "В фильме нет звука",
    "Туалет грязный ужасно",
    "Запах в туалете невыносимый",
    "Пыль везде летает, не чистили?",
    "Жарко в салоне просто невозможно",
    "Кресло очень неудобное",
    "Я попросил помощь, никто не пришёл",
    "Мой сосед развалился на моё кресло",
    "Меню вегетарианское есть?",
    "Детское питание заказывал, нет его",
    "Я без глютена, есть что?",
    "Напиток горячий дайте",
    "Спиртное есть на борту?",
    "Кто-то ворует мою ручную кладь",
    "Полка переполнена, мой багаж не влезает",
    "Мою сумку кто-то сломал",
    "Соседка всё время толкает мне в спину",
    "Может быть открыто окно?",
    "Кондиционер работает на полную",
    "Шум двигателя очень громкий",
    "Почему люди вокруг кашляют?",
    "Я боюсь, помогите пожалуйста",
    "Ребёнок плачет рядом уже час",
    "Свет не выключается в кресле",
    "Подлокотник сломан",
    "Карман в кресле порван",
    "Ремень безопасности не пристёгивается",
    "Я телезритель, покажи фильм какой?",
    "Музыка стоит очень громко",
    "Кино на русском есть?",
    "Это фильм уже 5 лет старый",
    "Документальные фильмы только?",
    "На русском языке нет ничего",
    "Интернет очень медленный",
    "Я не могу подключиться к Wi-Fi",
    "Пароль от Wi-Fi какой?",
    "Бортпроводник помог мне очень",
    "Спасибо за внимание!",
    "Дайте ещё подушку пожалуйста",
    "Я жарко одета, очень жарко",
    "Ещё одеяло хотелось бы",
    "Я болею, но летела заболев",

    # После вылета
    "Я хочу вернуть билет, вернёте ли деньги?",
    "Полёт был ужасен, требую возврата",
    "Я не летал, верните деньги",
    "Как поменять дату билета на более позднюю?",
    "Я хочу другой рейс вместо этого",
    "Требую компенсацию за задержку 5 часов",
    "Потеря багажа - сколько вы платите?",
    "Чемодан прибыл повреждённым",
    "Я требую 400 евро за задержку рейса",
    "EU261 компенсация когда приходит?",
    "Деньги за ваучер на отель кто возвращает?",
    "Я потратил 5000 руб на такси",
    "Еда в аэропорту платная была",
    "Я забыл очки на борту",
    "Потеряла телефон в самолёте",
    "Мой рюкзак остался в самолёте",
    "Где искать найденные вещи?",
    "Очки дорогие были, на сумму?",
    "Документы я оставил в кресле",
    "Как часто находят вещи?",
    "Возмещение за потерянные документы?",
    "Я хочу сменить обратный билет",
    "Вернуть билет на зарплату нельзя?",
    "У меня был no-show, деньги вернут?",
    "Возврат по условиям эконом тарифа",
    "Я летал но хочу вернуть билет",
    "Через сколько дней придут деньги?",
    "На какой счёт переводите возврат?",
    "Комиссия банка кто платит?",
    "Изменить даты возвратного билета",
    "Вместо возврата предложите рейс",
    "Я болел, это уважительная причина?",
    "Смерть в семье, бесплатный возврат?",
    "Стихийное бедствие - как быть?",
    "Компенсация от авиакомпании за что?",
    "Когда платят компенсацию EU261?",
    "На Lufthansa летел, они платили",
    "Как подать претензию по возврату?",
    "Какие документы нужны для возврата?",
    "Я требую телефонный звонок от руководителя",
    "Это была моя последняя поездка у вас",
    "Суд будет, берегитесь!",
    "Я недоволен всем абсолютно",
    "Я требую больше денег!",
    "Вам в суд я подам!",
    "Скажи мне кому жаловаться",
    "Я знаю свои права!",
    "Компенсация это закон!",
    "Неужели вы не знаете EU261?",
    "Все авиакомпании платят",
    "Я выслал вам письмо по почте",

    # Лояльность и продажи
    "У меня есть мили, как их потратить?",
    "Мои мили не начислились, почему?",
    "Баланс миль какой я могу узнать?",
    "Как зарегистрироваться в программе?",
    "Нужна ли карточка программы лояльности?",
    "Я Gold статус, какие льготы есть?",
    "Как получить Platinum статус?",
    "Мили истекают через год?",
    "Можно ли передать мили другому?",
    "Я хочу обменять мили на билет",
    "50000 миль на сколько бонусов?",
    "Мили сняли со счёта без разрешения",
    "Я делал покупку но не начислили",
    "Номер карты какой указывать?",
    "Акция сейчас есть какая?",
    "Билеты со скидкой есть сейчас?",
    "Промокод дайте если можно",
    "Где введу код скидки при покупке?",
    "Скидка сколько процентов обычно?",
    "Распродажа когда будет?",
    "Дешевле всего когда билеты?",
    "Early bird скидка есть?",
    "Last minute оффер есть?",
    "Black Friday скидка на билеты?",
    "Новый год какая скидка?",
    "Рождество акция есть?",
    "Летний чартер дешевле?",
    "Групповая скидка существует?",
    "Студент скидка есть?",
    "Пенсионеры скидку получают?",
    "Семейный тариф есть ли?",
    "Какой тариф мне подходит?",
    "Разница между эконом и комфортом?",
    "Эконом плюс стоит брать?",
    "Премиум класс какая стоимость?",
    "Гибкий тариф что это?",
    "Нерефундируемый билет дешевле?",
    "Я подписка хочу на письма",
    "Как отписаться от писем?",
    "Рассылка на почту какая полезная?",
    "Льготы при подписке какие?",
    "Мне нравится ваша программа",
    "Статус Silver кто получает?",
    "В каком аэропорту работает программа?",
    "Кредитная карта программы есть?",
    "Партнёры программы кто есть?",
    "В ресторане могу ли потратить мили?",
    "На отель обменять мили можно?",
    "Можно ли дарить мили?",
    "Истечение миль как работает?",

    # Технические вопросы
    "Сайт не загружается совсем",
    "Ошибка 404 при входе",
    "Сервер отвечает 503 ошибка",
    "Приложение постоянно крашится",
    "Я не могу войти под своим паролем",
    "Забыл пароль как сбросить?",
    "Приложение очень медленное работает",
    "На мобильнике не работает",
    "Обновление приложения помогло?",
    "Какая последняя версия приложения?",
    "Браузер показывает ошибку какую-то",
    "Платёж не прошёл но деньги списались",
    "Я заполнил форму но она не отправилась",
    "При регистрации выдаёт ошибку",
    "Посадочный талон не скачивается",
    "Email с билетом не пришел",
    "Письмо в спам попало может?",
    "Как ещё получить билет?",
    "Скриншот билета подойдет?",
    "QR код не сгенерировался",
    "Чат-бот не отвечает на вопросы",
    "Бот всегда неправильный ответ дает",
    "Бот понимает русский язык?",
    "Как пообщаться с человеком?",
    "Автоответчик ничего не помогает",
    "На сайте старая информация",
    "Цена на сайте не совпадает",
    "Рейс исчез из результатов поиска",
    "Поиск билетов как-то странно работает",
    "Фильтры не работают при поиске",
    "Я на Windows XP - будет работать?",
    "На iPad приложение работает?",
    "Android версия когда выйдет?",
    "iOS обновление какой версии нужно?",
    "Совместимость с какими браузерами?",
    "На Firefox работает ли?",
    "Chrome лучше чем Safari?",
    "Интернет медленный сайт тормозит",
    "VPN использую сайт не работает",
    "Прокси через какой лучше?",
    "SSL сертификат ошибка какая-то",
    "Куки файлы удалить помогает?",
    "Cache очистить нужно?",
    "Консоль браузера что показывает?",
    "Как включить JavaScript?",
    "Фирвол блокирует сайт может?",
    "Я получил письмо с подтверждением?",
    "Письмо потеряется или будет?",
    "Есть ли проверка почты по SMS?",

    # Обратная связь
    "Ваша авиакомпания ужасная, так больше нельзя!",
    "Я очень недоволен сервисом вообще",
    "Это просто издевательство над пассажирами",
    "Никогда больше у вас не полечу",
    "Друзьям не советую вашу авиакомпанию",
    "Я требую разговор с руководителем",
    "Это неприемлемо и незаконно!",
    "Я подам жалобу в Роспотребнадзор",
    "Это мой последний полёт с вами",
    "Вы меня оскорбили своим сервисом",
    "Я требую официальное разбирательство",
    "Пришлю вам письмо с претензией",
    "Мой адвокат свяжется с вами",
    "Я разместю отзыв везде",
    "Спасибо за отличный сервис!",
    "Бортпроводница была очень любезна",
    "Персонал помог мне очень благодарен",
    "Отлично проведён полёт, спасибо!",
    "Вы лучше остальных авиакомпаний",
    "Я вас рекомендую всем друзьям",
    "Хороший сервис оценил я по достоинству",
    "Спасибо за внимательное отношение",
    "Полёт прошёл гладко спасибо вам",
    "Рейс пунктуален как всегда",
    "Достаточно хороший уровень сервиса",
    "Можно добавить больше фильмов?",
    "Почему нет горячей еды?",
    "Включите русское кино пожалуйста",
    "Было бы хорошо если бы была ещё очка",
    "Может быть премиум WiFi?",
    "Дайте выбрать подушку получше",
    "Плед потолще было бы неплохо",
    "Горячее полотенце после полёта?",
    "Тёплый ужин вместо холодного?",
    "Больше выбор спецпитания надо",
    "Детское меню скудное",
    "Веганское питание хорошо",
    "Игры для детей на экране?",
    "Какой-то контент для подростков?",
    "Лучше кино выбирайте пожалуйста",
    "Почему нет российского сериала?",
    "Задержка рейса можно лучше объяснять",
    "Персонал должен быть вежливее",
    "Чистота в туалете оставляет желать",
    "Сиденья износились уже наверное",
    "Ремни безопасности старые наверное",
    "Как улучшить опыт полёта?",
    "Какие у вас планы развития?",
    "Что думаете о конкурентах?",

    # Сервисные
    "Привет!",
    "Здравствуйте!",
    "Добрый день!",
    "Добрый вечер!",
    "Доброе утро!",
    "Привет, как дела?",
    "Хеллоу!",
    "Здравствуй!",
    "Как ваши дела?",
    "Что нового?",
    "До свидания!",
    "До встречи!",
    "Пока!",
    "Спасибо за помощь!",
    "Спасибо и до встречи!",
    "Спасибо за ответ",
    "Пока, удачи!",
    "Хорошего дня!",
    "Удачи вам!",
    "Благодарю за внимание!",
    "До скорого!",
    "Пока, спасибо!",
    "Спасибо, пока!",
    "До встречи скоро!",
    "Спасибо товарищ!",
    "Спасибо друже!",
    "Большое спасибо!",
    "Огромное спасибо!",
    "Я благодарен!",
    "Благодарю вас!",
    "Спасибо за помощь ещё раз",
    "Очень вам благодарен!",
    "Хорошего вам дня!",
    "Хорошего вечера!",
    "Счастливо!",
    "Всего хорошего!",
    "Давайте попрощаемся",
    "До встречи друзья!",
    "Привет всем!",
    "Привет там!",
    "Ты здесь?",
    "Кто-нибудь там?",
    "Я тут",
    "Готов помочь",
    "Жду ответа",
    "Спасибо что помогли",
    "Спасибо за консультацию",
    "Спасибо за совет",
    "До встречи завтра!",
    "До встречи завтра утром!",
]
print(len(test_data))

batch_size = 30

for i in range(0, len(test_data), batch_size):

  batch = test_data[i:i+batch_size]

  result = classify_texts_batch(batch)
  print(result)

399


Adding requests:   0%|          | 0/30 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[{'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'Лояльность и продажи', 'level_2': ['Акции тарифы']}, {'level_1': 'После вылета', 'level_2': ['Возврат билета']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование', 'Оплата']}, {'level_1': 'До Вылета', 'level_2': ['Багаж (до вылета)']}, {'level_1': 'До Вылета', 'level_2': ['Документы']}, {'level_1': 'До Вылета', 'level_2': ['Документы']}, {'level_1': 'До Вылета', 'level_2': ['Оплата']}, {'level_1': 'До Вылета', 'level_2': ['Оплата']}, {'level_1': 'До Вылета', 'level_2': ['Оплата']}, {'level_1': 'До Вылета', 'level_2': ['Регистрация']}, {'level_1': 'До Вылета', 'level_2': ['Регистрация']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'Лояльность и продажи', 'level_2': ['Программа лояльности']}, {'level_1': 'Лояльность и продажи', 'lev

Adding requests:   0%|          | 0/30 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[{'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие', 'Прощание']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Прощание']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Прощание']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Прощание']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сер

Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[{'level_1': 'Сервисные', 'level_2': ['Приветствие']}, {'level_1': 'Другое', 'level_2': ['Другое']}, {'level_1': 'До Вылета', 'level_2': ['Бронирование']}, {'level_1': 'Обратная связь', 'level_2': ['Жалоба', 'Эскалация']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Благодарность']}, {'level_1': 'Сервисные', 'level_2': ['Прощание']}, {'level_1': 'Сервисные', 'level_2': ['Прощание']}]
